<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/1_DataEngineering_%26_FinbertSentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch pandas deep-translator tqdm

import pandas as pd
import torch
from transformers import pipeline
from deep_translator import GoogleTranslator
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_news = pd.read_csv('/content/GoogleNews_BBCA.csv')

finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=0 if torch.cuda.is_available() else -1)
translator = GoogleTranslator(source='id', target='en')

tqdm.pandas(desc="Translating & AI Scoring")

def get_translation_and_sentiment(text):
    try:
        english_text = translator.translate(str(text)[:500])
        result = finbert(english_text)[0]
        label = result['label']

        if label == 'positive':
            score = 1
        elif label == 'negative':
            score = -1
        else:
            score = 0

        return pd.Series([english_text, score])
    except Exception:
        # Returns an empty string for translation and 0 for score on failure
        return pd.Series(["", 0])

df_news[['translated_title', 'ai_sentiment']] = df_news['title'].progress_apply(get_translation_and_sentiment)

print("\n Individual Scoring Complete! Here are 5 random headlines and their scores:")
display(df_news[['date', 'title', 'translated_title', 'ai_sentiment']].sample(5))

print("\nExporting data to CSV")
export_path = '/content/BBCA_News_Sentiment_Scored.csv'
df_news.to_csv(export_path, index=False)
print(f"Data successfully exported to: {export_path}")

In [ ]:
# Adjust the date format for grouping
df_news['parsed_date'] = pd.to_datetime(df_news['date'], format='mixed', errors='coerce')
df_news['clean_date'] = df_news['parsed_date'].dt.strftime('%Y-%m-%d')

# Average Sentiment score per formatted date
df_sentiment = df_news.groupby('clean_date')['ai_sentiment'].mean().reset_index()
df_sentiment.columns = ['Date', 'Sentiment_Score']

df_sentiment.to_csv('BBCA_Daily_Sentiment_FinBERT.csv', index=False)

print("Here is the final Average Sentiment Score per day:")
display(df_sentiment.head(5))

In [ ]:
df_stock = pd.read_csv('/content/BBCA_Stock_History.csv')

# Standardize date format (Patching the Excel day/month swap issue)
df_stock['Date'] = pd.to_datetime(df_stock['Date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')

# Filtering for date range
df_stock = df_stock[(df_stock['Date'] >= '2025-01-01') & (df_stock['Date'] <= '2026-03-07')].copy()

# Rename columns to standard financial terms
df_stock.rename(columns={'Price': 'Close', 'Vol.': 'Volume'}, inplace=True)

def convert_volume(vol_str):
    if pd.isna(vol_str):
        return 0
    vol_str = str(vol_str).upper().replace(',', '')
    if 'M' in vol_str:
        return float(vol_str.replace('M', '')) * 1000000
    elif 'K' in vol_str:
        return float(vol_str.replace('K', '')) * 1000
    elif 'B' in vol_str:
        return float(vol_str.replace('B', '')) * 1000000000
    else:
        return float(vol_str)

print("Converting Volume format...")
df_stock['Volume'] = df_stock['Volume'].apply(convert_volume)

# Sort chronologically for the LSTM
df_stock = df_stock.sort_values('Date').reset_index(drop=True)
stock_clean = df_stock[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

print("Stock Data Cleaned and Sorted!")
display(stock_clean.head(3))

In [ ]:
print("Fusing Stock Data with Daily Sentiment Scores...")

# Left join: Match all active trading days with their respective news sentiment
master_df = pd.merge(stock_clean, df_sentiment, on='Date', how='left')

# Sliding window 1 for time lag
df['Sentiment_Score'] = df['Sentiment_Score'].shift(1).

# If there were no news articles on a trading day, default the sentiment to 0 (Neutral)
master_df['Sentiment_Score'] = master_df['Sentiment_Score'].fillna(0)

# Export the final Master Dataset
output_name = 'BBCA_Master_Dataset_BiLSTM.csv'
master_df.to_csv(output_name, index=False)

print(f"File saved as: {output_name}")
print(f"Total Trading Days: {len(master_df)}")
display(master_df.head())